
<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

# Part 2 — How can we solve this issue?

The flaw is **circular analysis**, also called **"double dipping"** or non-independence
error. The *same* 20 trial groups' data are used twice:

1. First to **select** the cells — we keep exactly those cells that happen to correlate
   positively with $P$ in this particular sample.
2. Then to **test** the correlation — we re-measure the correlation between $P$ and the average
   of those hand-picked cells.

Because the selection step cherry-picks cells whose *noise* happens to align with $P$, the
averaged cell group is guaranteed to correlate with $P$ in the same sample even when no true
relationship exists. The final correlation test is therefore not valid: its null distribution
is not the standard one, so the reported $p$ value is meaningless.

**Better (independent) workflows** — make sure selection and testing do not reuse the same
data in a way that manufactures the correlation. In the sections below we develop three:

- **Cross-validation / data splitting:** select the "responsive" cells using one subset of
  trial groups, then compute PRCA and test its correlation with $P$ on a *held-out* subset
  that played no part in selection.
- **Multiple-comparison correction (FDR):** rather than averaging and re-testing, ask how many
  *individual* cells are significantly correlated with $P$ after controlling the false
  discovery rate across all `Ncells` tests.

We first apply each of these to the null data (this section) to check that they bring the
false-positive rate back to the nominal level; later we confirm on real-effect data
(Part 3) that they still detect a genuine effect.

This notebook is self-contained: the shared building blocks from Part 1 are imported
from `utils.py`, so it can be run on its own.

</div>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from utils import (
    simulate_null_session,
    correlate_and_select,
    annotate_corr,
    train_test_PRCA,
    fdr_bh_cells,
)

# Set a seed so the notebook is reproducible (remove for fresh random draws)
rng = np.random.default_rng(0)

Ngroups, Ncells = 20, 200
mP, sP, sA = 75, 8, 1

# Recreate the Part 1 null session (behavior, activity, cell positions).
Perf, DA, CellXY = simulate_null_session(rng, Ngroups, Ncells, mP, sP, sA)

# Part 1 circular workflow on the null data 
r, p, SelectedCells = correlate_and_select(DA, Perf)
PRCA = DA[:, SelectedCells].mean(axis=1)



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

## Solution 1: a 50/50 train/test split

Break the circularity by using different sets of trial groups to select the cells and test the correlation.

* **Train half** (10 trial groups): select the responsive cells (same criteria, $r > 0.1$,
  $p < 0.05$).
* **Test half** (the other 10 trial groups, untouched during selection): compute PRCA on the
  selected cells and correlate it with $P$.

Because the test half played no part in selection, its correlation test is valid and the
`pearsonr` p-value is trustworthy. The cost is reduced power: selection and testing each use
only half the trial groups.

</div>


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">
<p><b>Task 2.1:</b> Implement <code>train_test_PRCA</code>: split the trial groups into a training half and a test half, select the responsive cells (<code>r &gt; 0.1</code> and <code>p &lt; 0.05</code>) using only the training half, then compute PRCA on the test half using those selected cells.
</div>


In [ ]:
def train_test_PRCA(DA, Perf, train_frac=0.5, generator=None):
    """Single train/test split: select cells on the training trial groups, then compute and
    return PRCA on the held-out test trial groups (which played no part in selection).

    Returns (PRCA_test, Perf_test, selected_cells), or None if no cells were selected.
    """
    # YOUR CODE HERE
    # 1. Split the Ngroups trial groups into a training index set and a test index set
    #    (e.g. train_frac of the groups for training, the rest for testing).
    # 2. Select cells using ONLY the training groups (same criteria: r > 0.1, p < 0.05).
    # 3. Compute PRCA on the TEST groups using the cells selected in step 2.
    # 4. Return (PRCA_test, Perf_test, selected_cells), or None if no cells were selected.
    pass


# Apply the 50/50 split to the null data (contiguous first-half / second-half split).
split_null = train_test_PRCA(DA, Perf)
if split_null is None:
    print("Null data: no cells selected on the training half.")
else:
    PRCA_test_null, Perf_test_null, sel_null = split_null
    r_tt, p_tt = stats.pearsonr(PRCA_test_null, Perf_test_null)
    print("Null data — 50/50 train/test split:")
    print(f"  cells selected on training half : {sel_null.size}")
    print(f"  test-half correlation           : r = {r_tt:+.3f}, p = {p_tt:.3f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    annotate_corr(ax, PRCA_test_null, Perf_test_null,
                  "Train/Test Split — No Real Effect\n(test half: independent of selection)")
    plt.tight_layout()
    plt.show()



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

### Comparison plot: circular analysis vs. train-test split

Two panels show the scatter of PRCA vs. behavioral performance $P$ on the **null** data.
On the left is the circular analysis (same trial groups used to select and to test); on the
right is train-test split (each trial group's PRCA comes from cells selected on the *other* trial
groups). In the train-test split plot only trial groups whose fold produced at least one selected cell
are shown.

**Expected outcome:** the circular panel shows a spurious significant correlation while the
train-test split panel shows a weak, non-significant one ($p \gg 0.05$).

</div>


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
annotate_corr(axes[0], PRCA, Perf,
              "Circular — No Real Effect\n(same data used for selection & test)")
annotate_corr(axes[1], PRCA_test_null, Perf_test_null,
                "Train/Test Split — No Real Effect\n(test half: independent of selection)")
plt.tight_layout()
plt.show()



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

## Solution 2: Multiple-comparison correction

The train-test split approach breaks the circularity by separating selection and testing. A different
valid strategy avoids the circular *average-and-retest* step altogether: rather than pooling
selected cells and re-testing their average, we can correct our selection criteria so we don't erroneously select 
cells that aren't truly correlated.

Viewed in this way, the problem is that if a single test has false positive rate 5% but we run it many times, the 
overall false positive rate increases and eventually approaches 100%. 

- **family-wise error rate (FWER)** correction controls the likelihood of *any* cell being falsely selected. A simple (conservative approach) is the **Bonferroni** procedure - simply multiply the p-values by the number of tests.
- **false discovery rate (FDR)** correction controls what percent of the selected cells are false positives (false discoveries). One approach is the **Benjamini-Hochberg** procedure, implemented in `scipy.stats.false_discovery_control`.

The number of significant cells is itself the read-out in this approach: if it is $\ge 1$ we would declare a
detected effect. Because each per-cell test uses the full data only *once* (no re-testing of a
selected average), this is not circular. We might plot the PRCA/performance relationship for visualization after, but that won't determine our conclusion.

</div>


<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">
<p><b>Task 2.2:</b> Write a function that takes the <code>DA</code> and <code>Perf</code> arrays, runs <code>pearsonr</code> for every cell, then returns the indices of cells matching our selection criteria (<code>r &gt; 0.1</code> and <code>p &lt; 0.05</code>) after correcting for multiple comparisons (FWER via Bonferroni, or FDR via Benjamini-Hochberg).
</div>


In [ ]:
def fwer_cells(DA, Perf, q=0.05):
    """Select cells whose correlation with Perf survives Bonferroni at level q.

    Returns the indices of the surviving cells (the FDR-significant discoveries).
    """
    # YOUR CODE HERE
    # 1. Run pearsonr for every cell against Perf.
    # 2. Apply the Bonferroni correction (multiply p-values by the number of cells).
    # 3. Return the indices of cells whose corrected p-value is <= q.
    pass


def fdr_bh_cells(DA, Perf, q=0.05):
    """Select cells whose correlation with Perf survives BH-FDR at level q.

    Returns the indices of the surviving cells (the FDR-significant discoveries).
    """
    # YOUR CODE HERE
    # 1. Run pearsonr for every cell against Perf.
    # 2. Apply Benjamini-Hochberg FDR correction (see scipy.stats.false_discovery_control).
    # 3. Return the indices of cells whose corrected p-value is <= q.
    pass


q_fdr = 0.05
fdr_null = fdr_bh_cells(DA, Perf, q=q_fdr)
print(f"FDR-BH (q = {q_fdr}) cells discovered on null data: {fdr_null.size}")

fwer_null = fwer_cells(DA, Perf, q=0.05)
print(f"FWER (Bonferroni, q = 0.05) cells discovered on null data: {fwer_null.size}")



<div style="border-left: 3px solid #000; padding: 1px; padding-left: 10px; background: #F0FAFF; ">

### False-positive rate — three methods

A single simulation run is not enough to measure the false-positive rate — one run
either passes or fails the significance threshold by chance.  To estimate the actual
false-positive rate we repeat the **null simulation** (no real effect) many times and
count how often each method incorrectly declares a significant result.

The loop below runs `N_iter = 1000` fresh null datasets.  For each:

1. Draw new `DA` (`Perf` can stay fixed or be resampled also).
2. Run the **circular** analysis and record whether the final $p < \alpha$.
3. Run the **train/test split** analysis and record whether the test-half $p < \alpha$.
4. Run the **multiple comparison** analysis and record whether it declares $\ge 1$ significant cell at $q = \alpha$.

At $\alpha = 0.05$ the nominal false-positive rate is 5%.

| Method | Selection / test | Trial groups used for selection | Trial groups used for final test |
|---|---|---|---|
| Circular | Correlation with $P$, average, re-test | All 20 | Same 20 |
| Train/test split | Correlation with $P$ | 10 (train half) | 10 (test half) |
| Multiple comparison | Per-cell correlation with $P$, corrected | All 20 | All 20 (no re-test of an average) |

**Expected outcome:** circular far above nominal; the train/test split and FDR-BH near
nominal.

*(Leave-one-out cross-validation — a more data-efficient valid method — is introduced in its
own section below, along with related pitfalls)*

</div>


In [ ]:
N_iter2 = 1000
alpha2 = 0.05

counts = {"circular": 0, "split": 0, "fdr_bh": 0}

for _ in range(N_iter2):
    DA_mc = sA * rng.standard_normal((Ngroups, Ncells))

    # Circular
    res_mc = stats.pearsonr(DA_mc, Perf[:, np.newaxis], axis=0)
    sel_mc = np.where((res_mc.statistic > 0.1) & (res_mc.pvalue < 0.05))[0]
    if sel_mc.size > 0:
        _, p_circ = stats.pearsonr(DA_mc[:, sel_mc].mean(axis=1), Perf)
        if p_circ < alpha2:
            counts["circular"] += 1

    # 50/50 train/test split
    tt = train_test_PRCA(DA_mc, Perf)
    if tt is not None and tt[0].size >= 3:
        _, p_tt = stats.pearsonr(tt[0], tt[1])
        if p_tt < alpha2:
            counts["split"] += 1

    # FDR-BH: declare an effect if >=1 cell survives BH at q = alpha2
    if fdr_bh_cells(DA_mc, Perf, q=alpha2).size > 0:
        counts["fdr_bh"] += 1

for name, cnt in counts.items():
    print(f"  {name:10s}  FP rate = {cnt/N_iter2:.1%}  ({cnt}/{N_iter2})")
print(f"  nominal α  = {alpha2:.1%}")


In [ ]:
labels = ["Circular\n(double-dipping)", "Train/Test\nsplit", "FDR-BH\n(corrected)"]
rates2 = [counts[k] / N_iter2 for k in ("circular", "split", "fdr_bh")]
colors2 = ["tab:red", "tab:blue", "tab:purple"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, rates2, color=colors2, width=0.5, edgecolor="k")
ax.axhline(alpha2, color="k", linestyle="--", linewidth=1.2, label=f"Nominal α = {alpha2}")
ax.set_ylabel("False-positive rate (null data)")
ax.set_title(f"False Positive rate at α = {alpha2}   (N = {N_iter2} simulations)")
ax.set_ylim(0, max(rates2) * 1.3)
for bar, rate in zip(bars, rates2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{rate:.1%}", ha="center", va="bottom", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()
